### XGBoost

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hmmlearn import hmm
from hmmlearn.hmm import GaussianHMM
import plotly.express as px
import mlflow
from mlflow.models.signature import infer_signature
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek
from scipy.stats import wilcoxon
from itertools import combinations


In [2]:
files = ['two_class_raw_1s_no.csv', 'two_class_raw_1s_yo_0.5.csv', 'two_class_raw_1s_yo_0.8.csv', 'two_class_raw_2s_no.csv', 'two_class_raw_2s_yo_0.5.csv', 'two_class_raw_2s_yo_0.8.csv', 
        'two_class_raw_3s_no.csv', 'two_class_raw_3s_yo_0.5.csv', 'two_class_raw_3s_yo_0.8.csv', 'two_class_raw_4s_no.csv', 'two_class_raw_4s_yo_0.5.csv', 'two_class_raw_4s_yo_0.8.csv',
        'two_class_raw_5s_no.csv', 'two_class_raw_5s_yo_0.5.csv', 'two_class_raw_5s_yo_0.8.csv']

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/improved_features_data_main/'

In [ ]:

def compare_sampling_techniques(X_train, y_train, groups_train, random_state=42):
    """
    Compare different sampling techniques for imbalanced classification
    """
    
    # Define sampling strategies to test
    sampling_strategies = {
        'none': None,
        'oversample_smote': SMOTE(random_state=random_state),
        'undersample_tomek_links': TomekLinks(),
        'combined_smote_tomek': SMOTETomek(random_state=random_state)
    }
    
    results = {}
    
    for strategy_name, sampler in sampling_strategies.items():
        print(f"\nTesting {strategy_name}...")
        
        # Cross-validation setup
        gkf = GroupKFold(n_splits=10) 
        fold_results = []
        
        for fold_num, (train_idx_fold, test_idx_fold) in enumerate(gkf.split(X_train, y_train, groups_train)):
            X_train_fold, X_test_fold = X_train.iloc[train_idx_fold], X_train.iloc[test_idx_fold]
            y_train_fold, y_test_fold = y_train.iloc[train_idx_fold], y_train.iloc[test_idx_fold]

            
            # Apply sampling (if any)
            if sampler is not None:
                try:
                    X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_fold, y_train_fold)
                    X_train_resampled = pd.DataFrame(X_train_resampled, columns=X_train_fold.columns)
                    y_train_resampled = pd.Series(y_train_resampled)
                except Exception as e:
                    print(f"Sampling failed for {strategy_name}: {e}")
                    continue
            else:
                X_train_resampled = X_train_fold
                y_train_resampled = y_train_fold
            
            # Use XGBoost with DEFAULT parameters
            model = XGBClassifier(
                random_state=random_state,
                eval_metric='logloss',  # Suppress warning
            )
            
            # Encode labels
            label_encoder = LabelEncoder()
            y_train_encoded = label_encoder.fit_transform(y_train_resampled)
            
            # Fit model
            model.fit(X_train_resampled, y_train_encoded)
            
            # Predict
            predictions = label_encoder.inverse_transform(model.predict(X_test_fold))
            
            # Calculate metrics
            accuracy = accuracy_score(y_test_fold, predictions)
            report_dict = classification_report(y_test_fold, predictions, output_dict=True)


            precision_void = report_dict.get("void", {}).get("precision", 0.0)
            recall_void = report_dict.get("void", {}).get("recall", 0.0)
            f1_void = report_dict.get("void", {}).get("f1-score", 0.0)

            precision_non_void = report_dict.get("non-void", {}).get("precision", 0.0)
            recall_non_void = report_dict.get("non-void", {}).get("recall", 0.0)
            f1_non_void = report_dict.get("non-void", {}).get("f1-score", 0.0)

            macro_f1 = report_dict.get("macro avg", {}).get("f1-score", 0.0)
            
            # Store detailed results
            fold_results.append({
                'fold': fold_num,
                'recall_void': recall_void,
                'precision_void': precision_void,
                'f1_void': f1_void,
                'macro_f1': macro_f1,
                'accuracy': accuracy,
                'precision_non_void': precision_non_void,  # Assuming 'non-void' is majority
                'recall_non_void': recall_non_void,
                'f1_non_void': f1_non_void,
                'class_distribution_train': dict(y_train_resampled.value_counts()),
                'class_distribution_test': dict(y_test_fold.value_counts())
            })
            

        # Calculate summary statistics
        if fold_results:  # Only if we have valid results
            results[strategy_name] = {
                'mean_accuracy': np.mean([f['accuracy'] for f in fold_results]),
                'std_accuracy': np.std([f['accuracy'] for f in fold_results]),
                'mean_recall_minority': np.mean([f['recall_void'] for f in fold_results]),
                'std_recall_minority': np.std([f['recall_void'] for f in fold_results]),
                'mean_f1_minority': np.mean([f['f1_void'] for f in fold_results]),
                'std_f1_minority': np.std([f['f1_void'] for f in fold_results]),
                'mean_f1_majority': np.mean([f['f1_non_void'] for f in fold_results]),
                'fold_details': fold_results
            }
    
    return results

In [ ]:
# file_results = {}
# for file in tqdm(files, desc="Producing results for different sampling techniques"):
#     data_path = os.path.join(base_path, file)
#     features = pd.read_csv(data_path)
#     features.drop(['center_time', 'start_time', 'end_time'], axis=1, inplace=True)
#     details = file.split('_')
#     exp_name = f"{details[3]}_{details[-1].replace('.csv', '')}"
#     print(f"Analysing {exp_name}")
    
#     # split data
#     X = features.drop(columns=['label', 'experiment_id'])
#     y = features['label']
#     groups = features['experiment_id']

#     splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
#     train_idx, test_idx = next(splitter.split(X, y, groups))

#     X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
#     y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
#     groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
    
#     file_results[exp_name] = compare_sampling_techniques(X_train, y_train, groups_train, 42)

In [5]:
# # pickle the file
# with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/imb_files/xgb_imb_152.pkl', 'wb') as f:
#     pickle.dump(file_results, f)

## Analyse data

### Fetch the void recall for all the sampling techniques.

In [ ]:
folds = ['fold 0', 'fold 1', 'fold 2', 'fold 3', 'fold 4', 'fold 5', 'fold 6', 'fold 7', 'fold 8', 'fold 9']

In [ ]:
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/imb_files/xgb_imb_152.pkl', 'rb') as f:
    xgb_results = pickle.load(f)

In [ ]:
def statistical_test(df):
    """
    Perform pairwise statistical comparisons between sampling techniques using Wilcoxon signed-rank test.
    
    This function compares the performance of different sampling techniques using:
    - Wilcoxon signed-rank test for statistical significance
    - Rank-biserial correlation for effect size
    - Mean improvement between techniques
    - Win/tie/loss counts
    - Bonferroni correction for multiple comparisons
    
    Parameters
    ----------
    df : pandas.DataFrame
        A DataFrame containing performance metrics (e.g., recall scores) for different 
        sampling techniques. Columns should represent techniques, rows represent folds/runs.
        
    Returns
    -------
    pandas.DataFrame
        A DataFrame containing comparison results with columns:
        - comparison: Pairwise comparison label
        - technique_A: First technique in comparison
        - technique_B: Second technique in comparison
        - p_value: Raw p-value from Wilcoxon test
        - statistic: Test statistic from Wilcoxon test
        - effect_size: Rank-biserial correlation effect size
        - mean_improvement: Mean difference (A - B)
        - A_wins: Count of folds where A outperformed B
        - ties: Count of ties
        - B_wins: Count of folds where B outperformed A
        - significant: Boolean indicating if comparison is significant after Bonferroni correction
    """
    
    techniques = ['none', 'oversample_smote', 'undersample_tomek_links', 'combined_smote_tomek']
    
    # Input validation
    if not all(tech in df.columns for tech in techniques):
        missing = [tech for tech in techniques if tech not in df.columns]
        raise ValueError(f"Input DataFrame missing columns for techniques: {missing}")
    
    # Store all the pairwise comparison results
    pairwise_results = []
    
    for technique_A, technique_B in combinations(techniques, 2):
        scores_A = df[technique_A].values
        scores_B = df[technique_B].values
        
        # Wilcoxon signed-rank test
        try:
            statistic, p_value = wilcoxon(scores_A, scores_B)
        except ValueError as e:
            raise ValueError(
                f"Wilcoxon test failed for {technique_A} vs {technique_B}: {str(e)}. "
                "This often occurs when there are no differences between the samples."
            )
        
        # Calculate effect size (rank-biserial correlation)
        diff = scores_A - scores_B
        effect_size = np.sum(np.sign(diff)) / len(diff)
        
        # Mean improvement
        mean_improvement = np.mean(diff)
        
        pairwise_results.append({
            'comparison': f"{technique_A} vs {technique_B}",
            'technique_A': technique_A,
            'technique_B': technique_B,
            'p_value': p_value,
            'statistic': statistic,
            'effect_size': effect_size,
            'mean_improvement': mean_improvement,
            'A_wins': np.sum(scores_A > scores_B),
            'ties': np.sum(scores_A == scores_B),
            'B_wins': np.sum(scores_B > scores_A)
        })
        
    # Convert to DataFrame
    comparison_df = pd.DataFrame(pairwise_results)

    # Apply Bonferroni correction
    total_comparisons = len(pairwise_results)
    corrected_alpha = 0.05 / total_comparisons
    comparison_df['significant'] = comparison_df['p_value'] < corrected_alpha

    return comparison_df

In [ ]:
# Ensure output directory exists
output_dir = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/XGBoost/2_class/imb_images'
os.makedirs(output_dir, exist_ok=True)

for win_data, data in xgb_results.items(): 
    recall_voids = {}
    
    for samp_tech in tqdm(data.keys(), desc=f"Processing {win_data}"):
        fold_details = data[samp_tech]['fold_details']
        metric = [fold['recall_void'] for fold in fold_details]
        recall_voids[samp_tech] = metric
    
    df = pd.DataFrame(recall_voids, index=folds)
    results = statistical_test(df)
    
    
    # Create figure with appropriate size
    fig, ax = plt.subplots(figsize=(30, 10))  # Increased width for better fit
    ax.axis('off')
    
    # Create table with auto-scale
    table = ax.table(
        cellText=results.values,
        colLabels=results.columns,
        # rowLabels=results.index,
        loc='center',
        cellLoc='center',
        colColours=['#f0f0f0']*len(results.columns)  # Light gray header
    )
    
    # Adjust layout and font size
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.2)  # Scale table
    
    # Add title
    plt.title(f"Statistical Test Results - {win_data}", pad=20)
    
    # Tight layout to prevent cutoff
    plt.tight_layout()
    
    # Save with high DPI and close figure
    output_path = os.path.join(output_dir, f"{win_data}.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()  # Important to free memory
    
    # Convert to csv
    output_path_csv = os.path.join(output_dir, f"{win_data}.csv")
    results.to_csv(output_path_csv)

In [48]:
df

,none,oversample_smote,undersample_tomek_links,combined_smote_tomek
fold 0,0.896552,0.931034,0.862069,0.931034
fold 1,0.590909,0.681818,0.681818,0.568182
fold 2,0.297872,0.212766,0.382979,0.170213
fold 3,0.483871,0.645161,0.516129,0.548387
fold 4,0.288462,0.423077,0.307692,0.346154
fold 5,0.490196,0.470588,0.529412,0.450980
fold 6,0.843137,0.823529,0.882353,0.862745
fold 7,0.454545,0.545455,0.600000,0.527273
fold 8,0.537313,0.552239,0.537313,0.537313
fold 9,0.518519,0.555556,0.518519,0.481481
